In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [ ]:
from pathlib import Path

DATA_DIR = Path("../Data/raw/datathon-2026-round-1")
OUTPUT_DIR = Path("../outputs")
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(DATA_DIR)
print(DATA_DIR.exists())

In [ ]:
sales = pd.read_csv(DATA_DIR / "sales.csv", parse_dates=["Date"])
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv", parse_dates=["Date"])

sales = sales.sort_values("Date").reset_index(drop=True)
sample_submission = sample_submission.sort_values("Date").reset_index(drop=True)

sales.head(), sample_submission.head()

In [ ]:
print(sales.info())
print()
print("Sales date range:", sales["Date"].min(), "->", sales["Date"].max())
print("Rows:", len(sales))
print()
print("Submission date range:", sample_submission["Date"].min(), "->", sample_submission["Date"].max())
print("Rows:", len(sample_submission))

In [37]:
print("Duplicate dates in sales:", sales["Date"].duplicated().sum())
print("Missing values in sales:")
print(sales.isna().sum())

print("\nDuplicate dates in sample_submission:", sample_submission["Date"].duplicated().sum())
print("Missing values in sample_submission:")
print(sample_submission.isna().sum())

Duplicate dates in sales: 0
Missing values in sales:
Date       0
Revenue    0
COGS       0
dtype: int64

Duplicate dates in sample_submission: 0
Missing values in sample_submission:
Date       0
Revenue    0
COGS       0
dtype: int64


In [ ]:
plt.figure(figsize=(14,5))
plt.plot(sales["Date"], sales["Revenue"])
plt.title("Daily Revenue Over Time")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.tight_layout()
plt.show()

In [ ]:
df_base = sales.copy()

df_base["pred_lag_1"] = df_base["Revenue"].shift(1)
df_base["pred_lag_7"] = df_base["Revenue"].shift(7)
df_base["pred_roll_7"] = df_base["Revenue"].shift(1).rolling(7).mean()

df_base[["Date", "Revenue", "pred_lag_1", "pred_lag_7", "pred_roll_7"]].head(15)

In [ ]:
valid_base = df_base.dropna().copy()

split_date = pd.Timestamp("2022-01-01")
train_base = valid_base[valid_base["Date"] < split_date]
test_base = valid_base[valid_base["Date"] >= split_date]

def evaluate_forecast(y_true, y_pred, name="model"):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return pd.DataFrame({
        "model": [name],
        "MAE": [mae],
        "RMSE": [rmse],
        "R2": [r2]
    })

baseline_results = pd.concat([
    evaluate_forecast(test_base["Revenue"], test_base["pred_lag_1"], "lag_1"),
    evaluate_forecast(test_base["Revenue"], test_base["pred_lag_7"], "lag_7"),
    evaluate_forecast(test_base["Revenue"], test_base["pred_roll_7"], "roll_mean_7"),
], ignore_index=True)

baseline_results.sort_values("RMSE")

In [ ]:
df = sales.copy()
df = df.sort_values("Date").reset_index(drop=True)

In [ ]:
df["year"] = df["Date"].dt.year
df["month"] = df["Date"].dt.month
df["quarter"] = df["Date"].dt.quarter
df["day"] = df["Date"].dt.day
df["dayofweek"] = df["Date"].dt.dayofweek
df["weekofyear"] = df["Date"].dt.isocalendar().week.astype(int)
df["dayofyear"] = df["Date"].dt.dayofyear
df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
df["is_month_start"] = df["Date"].dt.is_month_start.astype(int)
df["is_month_end"] = df["Date"].dt.is_month_end.astype(int)

df.head()

In [ ]:
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

df["dayofweek_sin"] = np.sin(2 * np.pi * df["dayofweek"] / 7)
df["dayofweek_cos"] = np.cos(2 * np.pi * df["dayofweek"] / 7)

df["dayofyear_sin"] = np.sin(2 * np.pi * df["dayofyear"] / 365.25)
df["dayofyear_cos"] = np.cos(2 * np.pi * df["dayofyear"] / 365.25)

In [ ]:
lag_list = [1, 7, 14, 28, 56, 365]

for lag in lag_list:
    df[f"rev_lag_{lag}"] = df["Revenue"].shift(lag)
    df[f"cogs_lag_{lag}"] = df["COGS"].shift(lag)

df.head(10)

In [ ]:
window_list = [7, 14, 28]

for window in window_list:
    df[f"rev_roll_mean_{window}"] = df["Revenue"].shift(1).rolling(window).mean()
    df[f"rev_roll_std_{window}"] = df["Revenue"].shift(1).rolling(window).std()
    df[f"rev_roll_min_{window}"] = df["Revenue"].shift(1).rolling(window).min()
    df[f"rev_roll_max_{window}"] = df["Revenue"].shift(1).rolling(window).max()

df.head(40)

In [ ]:
feature_df = df.dropna().reset_index(drop=True)

feature_df.shape
feature_df

In [ ]:
feature_cols = [
    "year", "month", "quarter", "day", "dayofweek", "weekofyear", "dayofyear",
    "is_weekend", "is_month_start", "is_month_end",
    "month_sin", "month_cos", "dayofweek_sin", "dayofweek_cos", "dayofyear_sin", "dayofyear_cos"
]

feature_cols += [f"rev_lag_{lag}" for lag in lag_list]
feature_cols += [f"cogs_lag_{lag}" for lag in lag_list]
feature_cols += [f"rev_roll_mean_{w}" for w in window_list]
feature_cols += [f"rev_roll_std_{w}" for w in window_list]
feature_cols += [f"rev_roll_min_{w}" for w in window_list]
feature_cols += [f"rev_roll_max_{w}" for w in window_list]

target_col = "Revenue"

len(feature_cols), feature_cols[:10]

In [ ]:
feature_df["Date"].min(), feature_df["Date"].max()
folds = [
    ("2020_fold", pd.Timestamp("2020-01-01"), pd.Timestamp("2021-01-01")),
    ("2021_fold", pd.Timestamp("2021-01-01"), pd.Timestamp("2022-01-01")),
    ("2022_fold", pd.Timestamp("2022-01-01"), pd.Timestamp("2023-01-01")),
]

In [ ]:
def train_and_evaluate_fold(train_df, valid_df, feature_cols, target_col):
    X_train = train_df[feature_cols]
    y_train = train_df[target_col]
    X_valid = valid_df[feature_cols]
    y_valid = valid_df[target_col]

    model = RandomForestRegressor(
        n_estimators=300,
        max_depth=12,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    preds = model.predict(X_valid)

    mae = mean_absolute_error(y_valid, preds)
    rmse = mean_squared_error(y_valid, preds) ** 0.5
    r2 = r2_score(y_valid, preds)

    return model, preds, {"MAE": mae, "RMSE": rmse, "R2": r2}

In [ ]:
fold_results = []
fold_predictions = {}

for fold_name, valid_start, valid_end in folds:
    train_df = feature_df[feature_df["Date"] < valid_start].copy()
    valid_df = feature_df[(feature_df["Date"] >= valid_start) & (feature_df["Date"] < valid_end)].copy()

    model, preds, metrics = train_and_evaluate_fold(train_df, valid_df, feature_cols, target_col)
    metrics["fold"] = fold_name
    fold_results.append(metrics)

    valid_out = valid_df[["Date", "Revenue"]].copy()
    valid_out["pred"] = preds
    fold_predictions[fold_name] = valid_out

cv_results = pd.DataFrame(fold_results)
cv_results

In [36]:
cv_results[["MAE", "RMSE", "R2"]].mean().to_frame("mean")

,mean
MAE,"542,464.7134"
RMSE,"769,330.0327"
R2,0.7825


In [ ]:
fold_to_plot = "2022_fold"
plot_df = fold_predictions[fold_to_plot]

plt.figure(figsize=(14,5))
plt.plot(plot_df["Date"], plot_df["Revenue"], label="Actual")
plt.plot(plot_df["Date"], plot_df["pred"], label="Predicted")
plt.title(f"Actual vs Predicted Revenue - {fold_to_plot}")
plt.xlabel("Date")
plt.ylabel("Revenue")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
X_full = feature_df[feature_cols]
y_full = feature_df[target_col]

final_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

final_model.fit(X_full, y_full)

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": final_model.feature_importances_
}).sort_values("importance", ascending=False)

importance_df.head(20)

In [ ]:
top_imp = importance_df.head(20)

plt.figure(figsize=(10,6))
sns.barplot(data=top_imp, x="importance", y="feature")
plt.title("Top 20 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
future_df = sample_submission[["Date"]].copy()
future_df["Revenue"] = np.nan
future_df["COGS"] = np.nan

history_df = df.copy()
history_df = history_df[["Date", "Revenue", "COGS"]].copy()

future_df.head()

In [ ]:
def build_features_for_date(history, current_date):
    temp = history.copy()
    row = {"Date": current_date}

    row["year"] = current_date.year
    row["month"] = current_date.month
    row["quarter"] = current_date.quarter
    row["day"] = current_date.day
    row["dayofweek"] = current_date.dayofweek
    row["weekofyear"] = int(current_date.isocalendar().week)
    row["dayofyear"] = current_date.dayofyear
    row["is_weekend"] = int(current_date.dayofweek >= 5)
    row["is_month_start"] = int(current_date.is_month_start)
    row["is_month_end"] = int(current_date.is_month_end)

    row["month_sin"] = np.sin(2 * np.pi * row["month"] / 12)
    row["month_cos"] = np.cos(2 * np.pi * row["month"] / 12)
    row["dayofweek_sin"] = np.sin(2 * np.pi * row["dayofweek"] / 7)
    row["dayofweek_cos"] = np.cos(2 * np.pi * row["dayofweek"] / 7)
    row["dayofyear_sin"] = np.sin(2 * np.pi * row["dayofyear"] / 365.25)
    row["dayofyear_cos"] = np.cos(2 * np.pi * row["dayofyear"] / 365.25)

    for lag in lag_list:
        row[f"rev_lag_{lag}"] = temp["Revenue"].iloc[-lag] if len(temp) >= lag else np.nan
        row[f"cogs_lag_{lag}"] = temp["COGS"].iloc[-lag] if len(temp) >= lag else np.nan

    for window in window_list:
        hist_rev = temp["Revenue"].iloc[-window:]
        row[f"rev_roll_mean_{window}"] = hist_rev.mean() if len(hist_rev) == window else np.nan
        row[f"rev_roll_std_{window}"] = hist_rev.std() if len(hist_rev) == window else np.nan
        row[f"rev_roll_min_{window}"] = hist_rev.min() if len(hist_rev) == window else np.nan
        row[f"rev_roll_max_{window}"] = hist_rev.max() if len(hist_rev) == window else np.nan

    return pd.DataFrame([row])

In [ ]:
history_forecast = history_df.copy()
future_preds = []

for current_date in future_df["Date"]:
    X_next = build_features_for_date(history_forecast, current_date)[feature_cols]
    pred_revenue = final_model.predict(X_next)[0]

    future_preds.append(pred_revenue)

    new_row = pd.DataFrame({
        "Date": [current_date],
        "Revenue": [pred_revenue],
        "COGS": [np.nan]
    })
    history_forecast = pd.concat([history_forecast, new_row], ignore_index=True)

future_df["Revenue"] = future_preds
future_df.head()

In [ ]:

X_full_cogs = feature_df[feature_cols]
y_full_cogs = feature_df["COGS"]

final_model_cogs = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)
final_model_cogs.fit(X_full_cogs, y_full_cogs)


history_forecast = history_df.copy()
future_revenue_preds = []
future_cogs_preds = []

for current_date in future_df["Date"]:
    X_next = build_features_for_date(history_forecast, current_date)[feature_cols]

    pred_revenue = final_model.predict(X_next)[0]
    pred_cogs = final_model_cogs.predict(X_next)[0]

    future_revenue_preds.append(pred_revenue)
    future_cogs_preds.append(pred_cogs)

    new_row = pd.DataFrame({
        "Date": [current_date],
        "Revenue": [pred_revenue],
        "COGS": [pred_cogs]
    })

    history_forecast = pd.concat([history_forecast, new_row], ignore_index=True)

future_df["Revenue"] = future_revenue_preds
future_df["COGS"] = future_cogs_preds
future_df.head()


In [ ]:
submission = sample_submission.copy()
submission["Revenue"] = future_df["Revenue"].values
submission["COGS"] = future_df["COGS"].values

submission.head(), submission.shape

In [ ]:
submission.to_csv(TABLE_DIR / "submission.csv", index=False)
print("Saved to:", TABLE_DIR / "submission.csv")

In [ ]:
print(submission.isna().sum())
print(submission.head())
print(submission.tail())